In [ ]:
!pip install -q gradio

In [4]:
import gradio as gr
import librosa
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

# 1. Architecture Setup
class TransferLearningMusicModel(nn.Module):
    def __init__(self):
        super(TransferLearningMusicModel, self).__init__()
        self.resnet = models.resnet18(weights=None)
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_features, 10)

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        return self.resnet(x)

# 2. Setup Device and Load Model
device = torch.device("cpu")
model = TransferLearningMusicModel().to(device)
# Ensure your model file is uploaded to the Colab file explorer
model.load_state_dict(torch.load("resnet_music_genre.pth", map_location=device))
model.eval()

genres = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

# 3. Enhanced Prediction Function: 5-Slice Averaging
def predict_genre(audio_path):
    if audio_path is None:
        return "No audio uploaded."

    # Load audio
    y_full, sr = librosa.load(audio_path)
    total_duration = librosa.get_duration(y=y_full, sr=sr)

    # Generate 5 starting points
    chunk_duration = 3.0
    num_chunks = 5
    if total_duration <= chunk_duration:
        start_times = [0.0] * num_chunks
    else:
        max_start = total_duration - chunk_duration
        start_times = np.linspace(0, max_start, num_chunks)

    tensors = []

    # Preprocess each slice
    for start in start_times:
        start_sample = int(start * sr)
        end_sample = start_sample + int(chunk_duration * sr)
        y_slice = y_full[start_sample:end_sample]

        mel_spec = librosa.feature.melspectrogram(y=y_slice, sr=sr, n_mels=128)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

        # Ensure dimensions [1, 128, 129]
        if mel_db.shape[1] > 129:
            mel_db = mel_db[:, :129]
        else:
            pad_width = 129 - mel_db.shape[1]
            mel_db = np.pad(mel_db, ((0, 0), (0, pad_width)), mode='constant')

        tensors.append(torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0))

    # Batch Predict
    batch_tensor = torch.stack(tensors).to(device)
    with torch.no_grad():
        outputs = model(batch_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        avg_probs = probabilities.mean(dim=0)

    return {genres[i]: float(avg_probs[i]) for i in range(len(genres))}

# 4. Launch App
demo = gr.Interface(
    fn=predict_genre,
    inputs=gr.Audio(type="filepath", label="Upload Music File"),
    outputs=gr.Label(num_top_classes=3, label="Top 3 Predicted Genres"),
    title=" Music Genre Classifier Pro",
    description="This model samples 5 segments from your song to ensure a highly accurate genre prediction."
)

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://40b3c4977b293a1be4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://40b3c4977b293a1be4.gradio.live


In [ ]:
from google.colab import drive
drive.mount('/content/drive')